In [17]:
# Imports
from pyspark.sql import functions as F
from pyspark.sql import types as T
from delta.tables import DeltaTable
import pandas as pd
from datetime import date, timedelta

print("Libraries loaded")


StatementMeta(, c57eb579-1abf-42d2-bed3-fca02d24d49e, 19, Finished, Available, Finished, False)

Libraries loaded


In [18]:
#  Build dim_date
# Generate dates from 2020 to 2027
# This is NOT from your data — it's generated manually
# Every analytics platform needs this

start = date(2020, 1, 1)
end   = date(2027, 12, 31)

rows = []
current = start
while current <= end:
    rows.append({
        "date_key":       int(current.strftime("%Y%m%d")),
        "full_date":      current,
        "year":           current.year,
        "quarter":        (current.month - 1) // 3 + 1,
        "month_num":      current.month,
        "month_name":     current.strftime("%B"),
        "month_short":    current.strftime("%b"),
        "week_of_year":   current.isocalendar()[1],
        "day_of_month":   current.day,
        "day_of_week":    current.isoweekday(),
        "day_name":       current.strftime("%A"),
        "is_weekend":     current.isoweekday() >= 6,
        "is_month_start": current.day == 1,
        "fiscal_year":    current.year if current.month >= 4 else current.year - 1,
        "fiscal_quarter": ((current.month - 4) % 12) // 3 + 1,
        "year_month":     int(current.strftime("%Y%m")),
    })
    current += timedelta(days=1)

dim_date_df = spark.createDataFrame(pd.DataFrame(rows))

# Silver Lakehouse Delta path
GOLD_DIM_DATE_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "gold_lakehouse.Lakehouse/Tables/dim_date"
)

(
    dim_date_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_DIM_DATE_PATH)
)

print("dim_date:", dim_date_df.count(), "rows")

StatementMeta(, c57eb579-1abf-42d2-bed3-fca02d24d49e, 20, Finished, Available, Finished, False)

dim_date: 2922 rows


In [19]:
# dim_customer
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_CUSTOMER_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "silver_lakehouse.Lakehouse/Tables/silver_customers"
)

GOLD_DIM_CUSTOMER_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "gold_lakehouse.Lakehouse/Tables/dim_customer"
)

# Read Silver table
customers_df = spark.read.format("delta").load(SILVER_CUSTOMER_PATH)

# Window for surrogate key
window_spec = Window.orderBy("customer_id")

# Build Dimension
dim_customer_df = (
    customers_df
    .filter(F.col("customer_id").isNotNull())
    .withColumn("customer_key", F.row_number().over(window_spec))
    .withColumn("full_name", F.concat_ws(" ", "first_name", "last_name"))
    .withColumn(
        "tenure_segment",
        F.when(F.col("days_as_customer") > 730, "Long-term")
         .when(F.col("days_as_customer") > 180, "Established")
         .when(F.col("days_as_customer") > 30, "Growing")
         .otherwise("New")
    )
    .withColumn("gold_created_at", F.current_timestamp())
)

# Write to Gold Lakehouse
(
    dim_customer_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_DIM_CUSTOMER_PATH)
)

print(f"dim_customer: {dim_customer_df.count()} rows")

StatementMeta(, c57eb579-1abf-42d2-bed3-fca02d24d49e, 21, Finished, Available, Finished, False)

dim_customer: 500 rows


In [20]:
# dim_geograpghy
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Silver Orders path
SILVER_ORDERS_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "silver_lakehouse.Lakehouse/Tables/silver_orders"
)

# Gold dim_geography path
GOLD_DIM_GEOGRAPHY_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "gold_lakehouse.Lakehouse/Tables/dim_geography"
)

# Read Silver Orders
orders_df = spark.read.format("delta").load(SILVER_ORDERS_PATH)

# Create distinct geography
geography_df = (
    orders_df
    .filter(F.col("city") != "Unknown")
    .select("city", "state")
    .distinct()
)

# Create surrogate key
window_spec = Window.orderBy("city", "state")

dim_geography_df = (
    geography_df
    .withColumn("geography_key", F.row_number().over(window_spec))
    .withColumn(
        "state_name",
        F.when(F.col("state") == "MH", "Maharashtra")
         .when(F.col("state") == "DL", "Delhi")
         .when(F.col("state") == "KA", "Karnataka")
         .when(F.col("state") == "TS", "Telangana")
         .when(F.col("state") == "TN", "Tamil Nadu")
         .when(F.col("state") == "GJ", "Gujarat")
         .when(F.col("state") == "RJ", "Rajasthan")
         .when(F.col("state") == "WB", "West Bengal")
         .otherwise("Other")
    )
    .withColumn(
        "region",
        F.when(F.col("state").isin("MH", "GJ"), "West")
         .when(F.col("state").isin("DL", "RJ"), "North")
         .when(F.col("state").isin("KA", "TS", "TN"), "South")
         .when(F.col("state") == "WB", "East")
         .otherwise("Other")
    )
    .withColumn(
        "city_tier",
        F.when(
            F.col("city").isin(
                "Mumbai",
                "Delhi",
                "Bangalore",
                "Hyderabad",
                "Chennai",
                "Kolkata"
            ),
            "Tier 1"
        ).otherwise("Tier 2")
    )
    .select(
        "geography_key",
        "city",
        "state",
        "state_name",
        "region",
        "city_tier"
    )
)

# Write to Gold Lakehouse
(
    dim_geography_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_DIM_GEOGRAPHY_PATH)
)

print(f"dim_geography: {dim_geography_df.count()} rows")

StatementMeta(, c57eb579-1abf-42d2-bed3-fca02d24d49e, 22, Finished, Available, Finished, False)

dim_geography: 60 rows


In [21]:
# dim_payment
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Silver Orders path
SILVER_ORDERS_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "silver_lakehouse.Lakehouse/Tables/silver_orders"
)

# Gold dim_payment path
GOLD_DIM_PAYMENT_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "gold_lakehouse.Lakehouse/Tables/dim_payment"
)

# Read Silver Orders
orders_df = spark.read.format("delta").load(SILVER_ORDERS_PATH)

# Window for surrogate key
window_spec = Window.orderBy("payment_method")

# Build dim_payment
dim_payment_df = (
    orders_df
    .filter(F.col("payment_method").isNotNull())
    .select("payment_method")
    .distinct()
    .withColumn("payment_key", F.row_number().over(window_spec))
    .withColumn(
        "payment_category",
        F.when(F.col("payment_method") == "UPI", "Digital")
         .when(F.col("payment_method") == "Credit Card", "Card")
         .when(F.col("payment_method") == "Debit Card", "Card")
         .when(F.col("payment_method") == "Net Banking", "Digital")
         .when(F.col("payment_method") == "Wallet", "Digital")
         .when(F.col("payment_method") == "Cash on Delivery", "Cash")
         .otherwise("Other")
    )
    .withColumn(
        "is_digital",
        F.col("payment_method").isin(
            "UPI",
            "Credit Card",
            "Debit Card",
            "Net Banking",
            "Wallet"
        )
    )
    .withColumn(
        "is_cod",
        F.col("payment_method") == "Cash on Delivery"
    )
    .select(
        "payment_key",
        "payment_method",
        "payment_category",
        "is_digital",
        "is_cod"
    )
)

# Write to Gold Lakehouse
(
    dim_payment_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_DIM_PAYMENT_PATH)
)

print(f"dim_payment: {dim_payment_df.count()} rows")

StatementMeta(, c57eb579-1abf-42d2-bed3-fca02d24d49e, 23, Finished, Available, Finished, False)

dim_payment: 6 rows


In [22]:
# Build fact_orders

# Silver Path
SILVER_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "silver_lakehouse.Lakehouse/Tables/"
)

# Gold Path
GOLD_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "gold_lakehouse.Lakehouse/Tables/"
)

# Read Delta tables and create temporary views
spark.read.format("delta").load(SILVER_PATH + "silver_orders") \
    .createOrReplaceTempView("silver_orders")

spark.read.format("delta").load(GOLD_PATH + "dim_date") \
    .createOrReplaceTempView("dim_date")

spark.read.format("delta").load(GOLD_PATH + "dim_customer") \
    .createOrReplaceTempView("dim_customer")

spark.read.format("delta").load(GOLD_PATH + "dim_geography") \
    .createOrReplaceTempView("dim_geography")

spark.read.format("delta").load(GOLD_PATH + "dim_payment") \
    .createOrReplaceTempView("dim_payment")

# Create fact_orders DataFrame
fact_orders_df = spark.sql("""
SELECT
    CAST(ROW_NUMBER() OVER (ORDER BY so.order_id) AS BIGINT) AS order_key,

    dd.date_key,
    dc.customer_key,
    dg.geography_key,
    dp.payment_key,

    so.order_id,
    so.customer_id,

    so.order_amount AS gross_revenue,
    so.item_count,

    CASE
        WHEN so.order_status = 'returned'
        THEN so.order_amount
        ELSE 0
    END AS returned_amount,

    CASE
        WHEN so.order_status = 'cancelled'
        THEN 1
        ELSE 0
    END AS is_cancelled,

    CASE
        WHEN so.order_status = 'delivered'
        THEN so.order_amount
        ELSE 0
    END AS net_revenue,

    CASE
        WHEN so.discount_code != 'NO_PROMO'
        THEN 1
        ELSE 0
    END AS has_discount,

    so.order_status,
    so.is_weekend,
    so.is_guest_order,
    so.order_hour,

    CURRENT_TIMESTAMP() AS gold_created_at

FROM silver_orders so

JOIN dim_date dd
    ON dd.date_key = CAST(DATE_FORMAT(so.order_date,'yyyyMMdd') AS INT)

LEFT JOIN dim_customer dc
    ON dc.customer_id = so.customer_id

LEFT JOIN dim_geography dg
    ON dg.city = so.city
   AND dg.state = so.state

LEFT JOIN dim_payment dp
    ON dp.payment_method = so.payment_method

WHERE so.order_id IS NOT NULL
  AND so.order_amount IS NOT NULL
""")

# Write to Gold Lakehouse
(
    fact_orders_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_PATH + "fact_orders")
)

print(f"fact_orders: {fact_orders_df.count()} rows")

StatementMeta(, c57eb579-1abf-42d2-bed3-fca02d24d49e, 24, Finished, Available, Finished, False)

fact_orders: 1400 rows


In [23]:
# Build agg_daily_sales

# Gold Lakehouse path
GOLD_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "gold_lakehouse.Lakehouse/Tables/"
)

# Read Gold tables and create temporary views
spark.read.format("delta").load(GOLD_PATH + "fact_orders") \
    .createOrReplaceTempView("fact_orders")

spark.read.format("delta").load(GOLD_PATH + "dim_date") \
    .createOrReplaceTempView("dim_date")

spark.read.format("delta").load(GOLD_PATH + "dim_geography") \
    .createOrReplaceTempView("dim_geography")

# Create aggregated DataFrame
agg_daily_sales_df = spark.sql("""
SELECT
    dd.full_date,
    dd.year,
    dd.quarter,
    dd.month_num,
    dd.month_name,
    dd.day_name,
    dd.is_weekend,
    dg.city,
    dg.state_name,
    dg.region,
    dg.city_tier,

    -- Revenue
    SUM(fo.gross_revenue) AS total_revenue,
    SUM(fo.net_revenue) AS net_revenue,
    SUM(fo.returned_amount) AS total_returns,

    -- Volume
    COUNT(fo.order_key) AS total_orders,
    SUM(fo.item_count) AS total_items,
    SUM(fo.is_cancelled) AS cancelled_orders,

    -- Averages
    ROUND(AVG(fo.gross_revenue), 2) AS avg_order_value,

    -- Rates
    ROUND(SUM(fo.is_cancelled) * 100.0 / COUNT(*), 2) AS cancellation_rate,
    ROUND(SUM(fo.has_discount) * 100.0 / COUNT(*), 2) AS discount_rate

FROM fact_orders fo
JOIN dim_date dd
    ON dd.date_key = fo.date_key
JOIN dim_geography dg
    ON dg.geography_key = fo.geography_key

GROUP BY
    dd.full_date,
    dd.year,
    dd.quarter,
    dd.month_num,
    dd.month_name,
    dd.day_name,
    dd.is_weekend,
    dg.city,
    dg.state_name,
    dg.region,
    dg.city_tier
""")

# Write to Gold Lakehouse
(
    agg_daily_sales_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_PATH + "agg_daily_sales")
)

print(f"agg_daily_sales: {agg_daily_sales_df.count()} rows")

StatementMeta(, c57eb579-1abf-42d2-bed3-fca02d24d49e, 25, Finished, Available, Finished, False)

agg_daily_sales: 980 rows


In [24]:
# Final verification of all Gold tables

GOLD_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "gold_lakehouse.Lakehouse/Tables/"
)

tables = [
    "dim_date",
    "dim_customer",
    "dim_geography",
    "dim_payment",
    "fact_orders",
    "agg_daily_sales"
]

print("=" * 50)
print("GOLD LAYER SUMMARY")
print("=" * 50)

for table in tables:
    df = spark.read.format("delta").load(GOLD_PATH + table)
    print(f"{table:<20} {df.count():>8} rows")

print("=" * 50)
print("Gold Layer Completed Successfullys!")

StatementMeta(, c57eb579-1abf-42d2-bed3-fca02d24d49e, 26, Finished, Available, Finished, False)

GOLD LAYER SUMMARY
dim_date                 2922 rows
dim_customer              500 rows
dim_geography              60 rows
dim_payment                 6 rows
fact_orders              1400 rows
agg_daily_sales           980 rows
Gold Layer Completed Successfullys!
